---

## بخش ۵ — مدل‌سازیِ طبقه‌بندی روی دادهٔ بانکی

پیش‌بینی پذیرش سپردهٔ مدت‌دار (`y`) پیش از تماس با مشتری.
- جداکنندهٔ فایل `;` است و **ترتیب ردیف‌ها زمانی** است (۲۰۰۸ تا ۲۰۱۰).
- `duration` فقط پس از پایان تماس معلوم می‌شود، پس هدف پیش‌بینی را نشت می‌دهد و حذف می‌شود.
- تقسیم ۷۰/۱۵/۱۵ با حفظ ترتیب زمانی، بدون بُر زدن.

In [ ]:
import json
import os
import warnings

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore')
RANDOM_STATE = 42

DATA = '../Day2_Datasets/01_BankMarketing/bank-additional-full.csv'
OUT = 'out'
os.makedirs(OUT, exist_ok=True)

## ۵.۱ شناخت داده: ابعاد، توازن کلاس‌ها، unknown و دو ستون تله

In [ ]:
df = pd.read_csv(DATA, sep=';')
print('shape:', df.shape)

y = (df['y'].astype(str).str.strip().str.lower() == 'yes').astype(int)
pos_rate = y.mean()
print(f'positive: {int(y.sum())} / {len(y)} = {100 * pos_rate:.4f}%')

cat_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c]) and c != 'y']
unknown_counts = {
    c: int((df[c].astype(str).str.strip() == 'unknown').sum()) for c in cat_cols
}
unknown_counts = {k: v for k, v in unknown_counts.items() if v > 0}
print('unknown:', unknown_counts)

pdays_999 = int((df['pdays'] == 999).sum())
print(f'pdays == 999: {pdays_999} rows ({100 * pdays_999 / len(df):.2f}%)')

dur = df.assign(y=y).groupby('y')['duration']
print('duration mean/median by class:',
      {int(k): (round(v.mean(), 1), float(v.median())) for k, v in dur})

`duration` در کلاس مثبت میانگین ۵۵۳ ثانیه و در منفی ۲۲۱ ثانیه است؛ یعنی حذف نکردنش
مدل را به «طول مکالمه» وابسته می‌کند که در لحظهٔ تصمیم (پیش از تماس) وجود ندارد.

`pdays = 999` یعنی «مشتری قبلاً هیچ‌وقت تماس نگرفته» — نه ۹۹۹ روز پیش. پس این مقدار
یک نشانگر گمشده است و باید به `NaN` + نشانگر دودویی تبدیل شود.

## ۵.۲ تقسیم زمانی و Pipeline

شش ویژگی مهندسی‌شده (همه بی‌حالت، بدون نشت):
`was_contacted_before`، `pdays_since`، `total_contacts`، `prev_success`،
`is_recession`، `age_bin`.

In [ ]:
def engineer(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    out['was_contacted_before'] = (out['pdays'] != 999).astype(int)
    out['pdays_since'] = out['pdays'].where(out['pdays'] != 999)      # 999 -> NaN
    out['total_contacts'] = out['campaign'] + out['previous']
    out['prev_success'] = (out['poutcome'].astype(str) == 'success').astype(int)
    out['is_recession'] = (out['emp.var.rate'] < 0).astype(int)
    out['age_bin'] = pd.cut(
        out['age'], bins=[16, 25, 35, 45, 55, 65, 120],
        labels=['<=25', '26-35', '36-45', '46-55', '56-65', '65+']).astype(object)
    out = out.drop(columns=['duration'])
    out = out.replace('unknown', np.nan)
    for c in out.columns:                      # keep plain object dtype for encoders
        if not pd.api.types.is_numeric_dtype(out[c]):
            out[c] = out[c].astype(object)
    return out


feat = engineer(df)
X = feat.drop(columns=['y'])
y = (feat['y'].astype(str).str.strip().str.lower() == 'yes').astype(int)

n = len(X)
i1, i2 = int(0.70 * n), int(0.85 * n)
X_tr, X_va, X_te = X.iloc[:i1], X.iloc[i1:i2], X.iloc[i2:]
y_tr, y_va, y_te = y.iloc[:i1], y.iloc[i1:i2], y.iloc[i2:]

for tag, Xs, ys in [('train', X_tr, y_tr), ('val', X_va, y_va), ('test', X_te, y_te)]:
    print(f'{tag:5s} n={len(Xs):5d}  pos={int(ys.sum()):5d}  rate={100 * ys.mean():5.2f}%')

num_cols = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
cat_cols = [c for c in X.columns if c not in num_cols]
print('numeric:', len(num_cols), '| categorical:', len(cat_cols))


def make_preprocessor(scale=True):
    num_steps = [('impute', SimpleImputer(strategy='median'))]
    if scale:
        num_steps.append(('scale', StandardScaler()))
    return ColumnTransformer(
        transformers=[
            ('num', Pipeline(num_steps), num_cols),
            ('cat', Pipeline([
                ('impute', SimpleImputer(strategy='most_frequent')),
                ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
            ]), cat_cols),
        ],
        remainder='drop',
        verbose_feature_names_out=False,
    )

### رانش شیوع کلاس در طول زمان

نرخ پذیرش از ~۳٪ (۲۰۰۸) به ~۴۶٪ (۲۰۱۰) می‌رسد؛ همبستگی `euribor3m` با شمارهٔ ردیف
‎−۰٫۸۵‎ است. پس شیوع کلاس در سه بخش عمداً متفاوت می‌شود و **PR-AUC بین بخش‌ها
قابل مقایسهٔ مستقیم نیست** (به شیوع وابسته است)؛ `ROC-AUC` این مشکل را ندارد.

## ۵.۳ مقایسهٔ مدل‌ها با اعتبارسنجی متناسب با زمان

In [ ]:
models = {
    'majority_baseline': Pipeline([
        ('prep', make_preprocessor()),
        ('clf', DummyClassifier(strategy='most_frequent')),
    ]),
    'logistic_regression': Pipeline([
        ('prep', make_preprocessor()),
        ('clf', LogisticRegression(max_iter=3000, solver='lbfgs', C=1.0,
                                   random_state=RANDOM_STATE)),
    ]),
    'random_forest': Pipeline([
        ('prep', make_preprocessor(scale=False)),
        ('clf', RandomForestClassifier(
            n_estimators=400, min_samples_leaf=100, max_features='sqrt',
            class_weight='balanced_subsample', n_jobs=1, random_state=RANDOM_STATE)),
    ]),
    'hist_gradient_boosting_extra': Pipeline([
        ('prep', make_preprocessor(scale=False)),
        ('clf', HistGradientBoostingClassifier(
            max_iter=200, learning_rate=0.1, max_leaf_nodes=4,
            min_samples_leaf=300, early_stopping=False, random_state=RANDOM_STATE)),
    ]),
}


def score_block(y_true, proba, thr=0.5):
    pred = (proba >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {
        'pr_auc': float(average_precision_score(y_true, proba)),
        'roc_auc': float(roc_auc_score(y_true, proba)),
        'f1_pos': float(f1_score(y_true, pred, pos_label=1)),
        'precision': float(precision_score(y_true, pred, pos_label=1, zero_division=0)),
        'recall': float(recall_score(y_true, pred, pos_label=1)),
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
        'cost': float(5 * fn + fp),
    }


tscv = TimeSeriesSplit(n_splits=5)
cv_results, val_results, train_results, val_proba = {}, {}, {}, {}
for name, pipe in models.items():
    rows = []
    for tr_idx, va_idx in tscv.split(X_tr):
        pipe.fit(X_tr.iloc[tr_idx], y_tr.iloc[tr_idx])
        rows.append(score_block(y_tr.iloc[va_idx],
                                pipe.predict_proba(X_tr.iloc[va_idx])[:, 1]))
    cv_results[name] = {
        k: float(np.mean([r[k] for r in rows])) for k in ['pr_auc', 'roc_auc', 'f1_pos']
    }
    cv_results[name]['pr_auc_std'] = float(np.std([r['pr_auc'] for r in rows]))
    cv_results[name]['fold_pos_rate'] = float(y_tr.mean())

    pipe.fit(X_tr, y_tr)
    val_proba[name] = pipe.predict_proba(X_va)[:, 1]
    train_results[name] = score_block(y_tr, pipe.predict_proba(X_tr)[:, 1])
    val_results[name] = score_block(y_va, val_proba[name])
    print(f'{name:28s} CV PR-AUC {cv_results[name]["pr_auc"]:.4f} '
          f'±{cv_results[name]["pr_auc_std"]:.4f} | '
          f'CV ROC-AUC {cv_results[name]["roc_auc"]:.4f} | '
          f'VAL PR-AUC {val_results[name]["pr_auc"]:.4f} '
          f'ROC-AUC {val_results[name]["roc_auc"]:.4f} '
          f'F1@.5 {val_results[name]["f1_pos"]:.4f}')

candidates = ['logistic_regression', 'random_forest', 'hist_gradient_boosting_extra']
best_name = max(candidates, key=lambda m: (val_results[m]['roc_auc'],
                                           val_results[m]['pr_auc']))
best_model = models[best_name]
print('\nselected:', best_name)

## ۵.۴ انتخاب آستانه با هزینهٔ نامتقارن (FN = ۵×FP)

In [ ]:
COST_FN, COST_FP = 5, 1


def cost_at(y_true, proba, thr):
    pred = (proba >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return float(COST_FN * fn + COST_FP * fp), (int(tn), int(fp), int(fn), int(tp))


pv = val_proba[best_name]
grid = np.round(np.arange(0.02, 0.96, 0.005), 3)
costs = np.array([cost_at(y_va, pv, t)[0] for t in grid])
best_thr = float(grid[int(np.argmin(costs))])
print(f'best threshold={best_thr:.3f} cost={costs.min():,.0f} '
      f'(at 0.5: {cost_at(y_va, pv, 0.5)[0]:,.0f})')

thr_report = {}
for thr in [0.5, best_thr]:
    cost, (tn, fp, fn, tp) = cost_at(y_va, pv, thr)
    pred = (pv >= thr).astype(int)
    thr_report[f'{thr:.3f}'] = {
        'threshold': float(thr), 'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp,
        'precision': float(precision_score(y_va, pred, pos_label=1, zero_division=0)),
        'recall': float(recall_score(y_va, pred, pos_label=1)),
        'f1': float(f1_score(y_va, pred, pos_label=1)),
        'cost': cost,
    }
    print(f'  thr={thr:.3f}: TN={tn} FP={fp} FN={fn} TP={tp} '
          f'P={thr_report[f"{thr:.3f}"]["precision"]:.3f} '
          f'R={thr_report[f"{thr:.3f}"]["recall"]:.3f} cost={cost:,.0f}')

## ۵.۵ اهمیت ویژگی‌ها روی اعتبارسنجی و نتیجهٔ نهایی روی آزمون

In [ ]:
perm = permutation_importance(best_model, X_va, y_va, scoring='roc_auc',
                              n_repeats=8, random_state=RANDOM_STATE, n_jobs=1)
imp = pd.Series(perm.importances_mean, index=X_va.columns).sort_values(ascending=False)
print(imp.head(10).round(4).to_string())

pos_imp = imp.clip(lower=0)
print(f'top-10 share of positive importance: '
      f'{100 * pos_imp.head(10).sum() / pos_imp.sum():.1f}%')

In [ ]:
pt = best_model.predict_proba(X_te)[:, 1]
test_block = score_block(y_te, pt, thr=best_thr)
test_at_05 = score_block(y_te, pt, thr=0.5)
print(f'TEST thr={best_thr:.3f}: PR-AUC {test_block["pr_auc"]:.4f} '
      f'ROC-AUC {test_block["roc_auc"]:.4f} F1 {test_block["f1_pos"]:.4f} '
      f'P {test_block["precision"]:.3f} R {test_block["recall"]:.3f} '
      f'cost {test_block["cost"]:,.0f}')
print(f'TEST @0.5 cost {test_at_05["cost"]:,.0f}')

## نمودارها

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for name in ['logistic_regression', 'random_forest', 'hist_gradient_boosting_extra']:
    p, r, _ = precision_recall_curve(y_va, val_proba[name])
    ax[0].plot(r, p, label=f'{name} (AP={val_results[name]["pr_auc"]:.3f})')
ax[0].axhline(y_va.mean(), ls='--', c='grey', label='prevalence (chance)')
ax[0].set(xlabel='Recall', ylabel='Precision', title='PR curve — validation')
ax[0].legend(fontsize=7)
ax[0].grid(alpha=.3)

ax[1].plot(grid, costs)
ax[1].axvline(best_thr, ls='--', c='r', label=f'best={best_thr:.2f}')
ax[1].axvline(0.5, ls=':', c='k', label='0.50')
ax[1].set(xlabel='threshold', ylabel='validation cost = 5·FN + FP',
          title='Asymmetric cost')
ax[1].legend(fontsize=8)
ax[1].grid(alpha=.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'sec5_pr_cost.png'), dpi=130)
plt.show()
plt.close()

fig, ax = plt.subplots(1, 2, figsize=(9, 3.8))
for k, thr in enumerate([0.5, best_thr]):
    cm = confusion_matrix(y_va, (pv >= thr).astype(int), labels=[0, 1])
    ax[k].imshow(cm, cmap='Blues')
    for i in range(2):
        for j in range(2):
            ax[k].text(j, i, f'{cm[i, j]:,}', ha='center', va='center',
                       color='white' if cm[i, j] > cm.max() / 2 else 'black')
    ax[k].set(title=f'validation @ thr={thr:.2f}', xticks=[0, 1], yticks=[0, 1],
              xlabel='predicted', ylabel='actual')
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'sec5_confusion.png'), dpi=130)
plt.show()
plt.close()

fig, ax = plt.subplots(figsize=(7, 4.2))
imp.head(12)[::-1].plot(kind='barh', ax=ax, color='#3b6ea5')
ax.set(xlabel='drop in ROC-AUC when permuted',
       title='Permutation importance — validation')
ax.grid(alpha=.3, axis='x')
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'sec5_importance.png'), dpi=130)
plt.show()
plt.close()

fig, ax = plt.subplots(figsize=(8, 3.6))
for name, series in [('train', y_tr), ('val', y_va), ('test', y_te)]:
    pass
rates = [100 * y_tr.mean(), 100 * y_va.mean(), 100 * y_te.mean()]
ax.bar(['train (2008-09)', 'val (2009-10)', 'test (2010)'], rates, color='#3b6ea5')
for i, v in enumerate(rates):
    ax.text(i, v + 1, f'{v:.1f}%', ha='center')
ax.set(ylabel='positive rate (%)', title='Chronological split: class-prevalence drift')
ax.grid(alpha=.3, axis='y')
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'sec5_drift.png'), dpi=130)
plt.show()
plt.close()
print('figures ->', OUT)

In [ ]:
results = {
    'n_rows': int(len(df)),
    'n_cols': int(df.shape[1]),
    'positive_count': int(y.sum()),
    'positive_rate_pct': round(100 * pos_rate, 4),
    'unknown_counts': unknown_counts,
    'pdays_999': pdays_999,
    'pdays_999_pct': round(100 * pdays_999 / len(df), 2),
    'split': {t: [len(s), round(100 * r.mean(), 2)]
              for t, s, r in [('train', X_tr, y_tr), ('val', X_va, y_va),
                              ('test', X_te, y_te)]},
    'engineered': ['was_contacted_before', 'pdays_since', 'total_contacts',
                   'prev_success', 'is_recession', 'age_bin'],
    'cv': cv_results,
    'train': train_results,
    'val': val_results,
    'selected_model': best_name,
    'threshold': {'best': best_thr, 'reported': thr_report},
    'test': test_block,
    'test_at_05': test_at_05,
    'importance': {k: float(v) for k, v in imp.head(12).items()},
}
with open(os.path.join(OUT, 'sec5_results.json'), 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print('saved out/sec5_results.json')

---

## بخش ۶ — رگرسیون و مهندسی ویژگی برای تقاضای دوچرخه

هدف: برآورد `cnt` (تعداد دوچرخهٔ اجاره‌شده) در هر ساعت از ویژگی‌های تقویمی و آب‌وهوایی همان ساعت.
برای کاربرد آینده‌نگر، هوا باید از پیش معلوم یا پیش‌بینی‌شده باشد؛ پس از هیچ مقدار آینده استفاده نمی‌کنیم.

In [ ]:
import json
import os
import time
import warnings

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore')
RANDOM_STATE = 42

DATA = '../Day2_Datasets/02_BikeSharing/hour.csv'
OUT = 'out'
os.makedirs(OUT, exist_ok=True)

## ۶.۱ مرتب‌سازی زمانی، تقسیم ۷۰/۱۵/۱۵ و خط پایه

In [ ]:
df = pd.read_csv(DATA, parse_dates=['dteday'])
df = df.sort_values(['dteday', 'hr']).reset_index(drop=True)
print('rows:', len(df), '| range:', df.dteday.min().date(), '->', df.dteday.max().date())

# لیکج obvious
print('casual + registered == cnt for all rows:',
      bool(((df.casual + df.registered) == df.cnt).all()))
print('corr(temp, atemp) = %.4f' % df.temp.corr(df.atemp))

n = len(df)
i1, i2 = int(0.70 * n), int(0.85 * n)
train, val, test = df.iloc[:i1], df.iloc[i1:i2], df.iloc[i2:]
print(f'train {len(train)} ({train.dteday.min().date()}..{train.dteday.max().date()})')
print(f'val   {len(val)} ({val.dteday.min().date()}..{val.dteday.max().date()})')
print(f'test  {len(test)} ({test.dteday.min().date()}..{test.dteday.max().date()})')

### خط پایه: میانگین `cnt` به تفکیک ساعت، فقط از آموزش

ساده‌ترین مدل «فصلی» ممکن: برای هر ساعت شبانه‌روز، میانگین تاریخی همان ساعت در آموزش.

In [ ]:
hour_profile = train.groupby('hr')['cnt'].mean()
baseline_val = val['hr'].map(hour_profile).to_numpy()
baseline_mae = mean_absolute_error(val['cnt'], baseline_val)
baseline_rmse = float(np.sqrt(mean_squared_error(val['cnt'], baseline_val)))
print(f'baseline (hour-of-day mean) VAL MAE={baseline_mae:.2f} RMSE={baseline_rmse:.2f}')

### ستون‌های نشت‌دار و بی‌فایده

| ستون | وضعیت | دلیل |
|---|---|---|
| `casual`، `registered` | **نشت اطلاعات** | اجزای تشکیل‌دهندهٔ هدف‌اند؛ `casual + registered = cnt` در تمام ردیف‌ها |
| `instant` | شناسهٔ بی‌فایده | شمارهٔ ترتیبی ردیف؛ حامل هیچ اطلاعاتی نیست |
| `dteday` | شناسه/شاخص زمان | برای مرتب‌سازی و تقسیم استفاده می‌شود، نه به‌عنوان ویژگی (وگرنه مدل روند زمانی را حفظ می‌کند) |
| `atemp` | هم‌خطی شدید | همبستگی ۰٫۹۸۸ با `temp`؛ نگه داشتن هر دو بی‌فایده و مضر است |

## ۶.۲ ویژگی‌های چرخه‌ای و تعاملی

- `hr_sin/hr_cos` با دورهٔ ۲۴، `mnth_sin/mnth_cos` با دورهٔ ۱۲، `weekday_sin/weekday_cos` با دورهٔ ۷
- تعاملی: `is_rush = workingday × (hr ∈ {7,8,9,17,18,19})` — چون اوج تقاضا در روزهای کاری
  ساعت ۸ و ۱۷–۱۸ است و در تعطیلات این اوج جابه‌جا می‌شود. بدون این تعامل، مدل خطی
  نمی‌تواند «شکل» اوج را بین روز کاری و تعطیل تفکیک کند.

In [ ]:
def engineer(frame: pd.DataFrame) -> pd.DataFrame:
    d = frame.copy()
    d['hr_sin'] = np.sin(2 * np.pi * d['hr'] / 24)
    d['hr_cos'] = np.cos(2 * np.pi * d['hr'] / 24)
    d['mnth_sin'] = np.sin(2 * np.pi * d['mnth'] / 12)
    d['mnth_cos'] = np.cos(2 * np.pi * d['mnth'] / 12)
    d['weekday_sin'] = np.sin(2 * np.pi * d['weekday'] / 7)
    d['weekday_cos'] = np.cos(2 * np.pi * d['weekday'] / 7)
    d['is_rush'] = ((d['workingday'] == 1) & d['hr'].isin([7, 8, 9, 17, 18, 19])).astype(int)
    d['work_hour_sin'] = d['workingday'] * d['hr_sin']
    d['work_hour_cos'] = d['workingday'] * d['hr_cos']
    d['weathersit'] = d['weathersit'].replace({4: 3})   # فقط ۳ ردیف؛ ادغام با «باران شدید»
    return d


feat = engineer(df)
DROP = ['instant', 'dteday', 'atemp', 'casual', 'registered', 'cnt']
X = feat.drop(columns=DROP)
y = feat['cnt']

X_tr, X_va, X_te = X.iloc[:i1], X.iloc[i1:i2], X.iloc[i2:]
y_tr, y_va, y_te = y.iloc[:i1], y.iloc[i1:i2], y.iloc[i2:]

num_cols = [c for c in X.columns if c not in ['season', 'weathersit']]
cat_cols = ['season', 'weathersit']
print('numeric (%d):' % len(num_cols), num_cols)
print('categorical:', cat_cols)

همهٔ تبدیل‌های نیازمند برازش (میانه/میانگین برای پر کردن، میانگین و انحراف‌معیار برای
استانداردسازی، و دسته‌های one-hot) داخل `Pipeline` هستند و فقط روی `X_tr` برازش می‌شوند.
هیچ آماره‌ای از val/test محاسبه نمی‌شود.

In [ ]:
def make_prep(scale=True):
    steps = [('impute', SimpleImputer(strategy='median'))]
    if scale:
        steps.append(('scale', StandardScaler()))
    return ColumnTransformer([
        ('num', Pipeline(steps), num_cols),
        ('cat', Pipeline([
            ('impute', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
        ]), cat_cols),
    ], remainder='drop', verbose_feature_names_out=False)

## ۶.۳ مقایسهٔ سه مدل با اعتبارسنجی زمانی

In [ ]:
models = {
    'dummy_median': Pipeline([
        ('prep', make_prep()),
        ('reg', DummyRegressor(strategy='median')),
    ]),
    'ridge': Pipeline([
        ('prep', make_prep()),
        ('reg', Ridge(alpha=1.0, random_state=RANDOM_STATE)),
    ]),
    'hist_gradient_boosting': Pipeline([
        ('prep', make_prep(scale=False)),
        ('reg', HistGradientBoostingRegressor(
            max_iter=400, learning_rate=0.06, max_leaf_nodes=31,
            min_samples_leaf=20, l2_regularization=1.0,
            early_stopping=False, random_state=RANDOM_STATE)),
    ]),
}


def metrics(y_true, y_pred):
    return {
        'mae': float(mean_absolute_error(y_true, y_pred)),
        'rmse': float(np.sqrt(mean_squared_error(y_true, y_pred))),
    }


tscv = TimeSeriesSplit(n_splits=5)
results, fitted = {}, {}
for name, pipe in models.items():
    t0 = time.perf_counter()
    cv_mae = []
    for tr_idx, va_idx in tscv.split(X_tr):
        pipe.fit(X_tr.iloc[tr_idx], y_tr.iloc[tr_idx])
        cv_mae.append(mean_absolute_error(
            y_tr.iloc[va_idx], pipe.predict(X_tr.iloc[va_idx])))
    cv_time = time.perf_counter() - t0

    t0 = time.perf_counter()
    pipe.fit(X_tr, y_tr)
    fit_time = time.perf_counter() - t0
    pred_va = pipe.predict(X_va)
    fitted[name] = pred_va

    m = metrics(y_va, pred_va)
    m['cv_mae'] = float(np.mean(cv_mae))
    m['cv_mae_std'] = float(np.std(cv_mae))
    m['fit_seconds'] = round(fit_time, 3)
    m['cv_seconds'] = round(cv_time, 2)
    results[name] = m
    print(f'{name:26s} VAL MAE={m["mae"]:6.2f} RMSE={m["rmse"]:6.2f} | '
          f'CV MAE={m["cv_mae"]:6.2f}±{m["cv_mae_std"]:.2f} | fit {fit_time:.2f}s')

best_name = min(['ridge', 'hist_gradient_boosting'], key=lambda k: results[k]['mae'])
best = models[best_name]
print('\nselected:', best_name)

### آزمایشِ بهبود — فقط روی اعتبارسنجی (آزمون دست‌نخورده می‌ماند)

`cnt` یک شمارش است و واریانسش با میانگین رشد می‌کند؛ تابع هدفِ خطای مربعی روی مقدار خام
ساعت‌های پرتقاضا را بیش از حد وزن می‌دهد. همان مدل با تابع هدفِ پواسون:

In [ ]:
poisson = Pipeline([
    ('prep', make_prep(scale=False)),
    ('reg', HistGradientBoostingRegressor(
        loss='poisson', max_iter=400, learning_rate=0.06, max_leaf_nodes=31,
        min_samples_leaf=20, l2_regularization=1.0,
        early_stopping=False, random_state=RANDOM_STATE)),
]).fit(X_tr, y_tr)
poisson_val = metrics(y_va, poisson.predict(X_va))
print(f'poisson variant VAL MAE={poisson_val["mae"]:.2f} RMSE={poisson_val["rmse"]:.2f} '
      f'(current: {results[best_name]["mae"]:.2f} / {results[best_name]["rmse"]:.2f})')

## ۶.۴ ارزیابی نهایی روی آزمون (فقط یک بار)

In [ ]:
pred_te = best.predict(X_te)
test_metrics = metrics(y_te, pred_te)
print(f'TEST MAE={test_metrics["mae"]:.2f} RMSE={test_metrics["rmse"]:.2f}')

resid = pred_te - y_te.to_numpy()
by_hour = pd.DataFrame({'hr': X_te['hr'].to_numpy(), 'err': resid,
                        'actual': y_te.to_numpy(), 'pred': pred_te})
hour_stats = by_hour.groupby('hr')['err'].agg(['mean', 'std'])
print('\nmean error by hour (pred - actual):')
print(hour_stats.round(1).to_string())

bins = pd.qcut(by_hour['actual'], 5, labels=['Q1', 'Q2', 'Q3', 'Q4', 'Q5'])
print('\nmean error by demand quintile:')
print(by_hour.groupby(bins, observed=True)['err'].agg(['mean', 'count']).round(1).to_string())
print('\nmean error by weathersit:')
print(by_hour.assign(ws=X_te['weathersit'].to_numpy())
      .groupby('ws')['err'].agg(['mean', 'count']).round(1).to_string())
print(f'\noverall bias (mean residual) = {resid.mean():.2f}')

## نمودارها

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(11, 6.4), sharex=False)
seg = slice(0, 400)
ax[0].plot(y_te.to_numpy()[seg], label='actual', lw=1.2, color='#1f3b57')
ax[0].plot(pred_te[seg], label='predicted', lw=1.2, color='#c0392b', alpha=.85)
ax[0].set(title='Test: actual vs predicted (first 400 hours)', ylabel='cnt')
ax[0].legend()
ax[0].grid(alpha=.3)

ax[1].scatter(y_te, pred_te, s=4, alpha=.25, color='#3b6ea5')
lims = [0, max(y_te.max(), pred_te.max()) * 1.02]
ax[1].plot(lims, lims, 'k--', lw=1)
ax[1].set(xlabel='actual cnt', ylabel='predicted cnt', title='Predicted vs actual — test')
ax[1].grid(alpha=.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'sec6_fit.png'), dpi=130)
plt.show()
plt.close()

fig, ax = plt.subplots(figsize=(9, 4))
ax.errorbar(hour_stats.index, hour_stats['mean'], yerr=hour_stats['std'],
            fmt='o-', capsize=3, color='#c0392b', ecolor='#999')
ax.axhline(0, color='k', lw=.8)
ax.set(xlabel='hour of day', ylabel='mean error (pred - actual)',
       title='Error by hour of day — test')
ax.grid(alpha=.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'sec6_error_hour.png'), dpi=130)
plt.show()
plt.close()

fig, ax = plt.subplots(figsize=(7, 4))
by_hour.groupby(bins, observed=True)['err'].agg(['mean', 'std']).pipe(
    lambda d: ax.bar(range(len(d)), d['mean'], yerr=d['std'], capsize=4, color='#3b6ea5'))
ax.axhline(0, color='k', lw=.8)
ax.set_xticks(range(5))
ax.set_xticklabels(['Q1\nlowest', 'Q2', 'Q3', 'Q4', 'Q5\nhighest'])
ax.set(xlabel='actual demand quintile', ylabel='mean error (pred - actual)',
       title='Error by demand level — test')
ax.grid(alpha=.3, axis='y')
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'sec6_error_demand.png'), dpi=130)
plt.show()
plt.close()
print('figures ->', OUT)

In [ ]:
imp = permutation_importance(best, X_va, y_va, scoring='neg_mean_absolute_error',
                             n_repeats=6, random_state=RANDOM_STATE, n_jobs=1)
imp_s = pd.Series(imp.importances_mean, index=X_va.columns).sort_values(ascending=False)
print('permutation importance (MAE increase):')
print(imp_s.head(8).round(2).to_string())

out = {
    'n_rows': int(len(df)),
    'split': {'train': len(train), 'val': len(val), 'test': len(test)},
    'date_ranges': {
        'train': [str(train.dteday.min().date()), str(train.dteday.max().date())],
        'val': [str(val.dteday.min().date()), str(val.dteday.max().date())],
        'test': [str(test.dteday.min().date()), str(test.dteday.max().date())],
    },
    'baseline': {'mae': float(baseline_mae), 'rmse': float(baseline_rmse)},
    'models': results,
    'selected': best_name,
    'improvement_poisson': poisson_val,
    'test': test_metrics,
    'bias_test': float(resid.mean()),
    'error_by_hour': {int(k): round(float(v), 1) for k, v in hour_stats['mean'].items()},
    'error_by_quintile': {str(k): round(float(v), 1) for k, v in
                          by_hour.groupby(bins, observed=True)['err'].mean().items()},
    'importance': {k: float(v) for k, v in imp_s.head(8).items()},
}
with open(os.path.join(OUT, 'sec6_results.json'), 'w', encoding='utf-8') as f:
    json.dump(out, f, indent=2, ensure_ascii=False)
print('saved out/sec6_results.json')

---

## بخش ۷ — شبکهٔ عصبی و طبقه‌بندی تصویر (Fashion-MNIST)

۱۰ کلاس پوشاک، تصاویر خاکستری ۲۸×۲۸. مجموعهٔ آزمون (۱۰٬۰۰۰ تصویر) تا مرحلهٔ نهایی
دست‌نخورده می‌ماند؛ همهٔ تصمیم‌ها روی اعتبارسنجی گرفته می‌شود.

In [ ]:
import json
import os
import sys
import time
import warnings

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.use_deterministic_algorithms(False)

DATA_DIR = '../Day2_Datasets/03_FashionMNIST'
sys.path.insert(0, DATA_DIR)
from load_fashion import CLASS_NAMES, load_fashion  # noqa: E402

OUT = 'out'
os.makedirs(OUT, exist_ok=True)
DEVICE = 'cpu'

## ۷.۱ شناخت داده، تقسیم آموزش/اعتبارسنجی و نرمال‌سازی

In [ ]:
x_all, y_all, x_test_raw, y_test_raw = load_fashion()
print('train images:', x_all.shape, x_all.dtype)
print('train labels:', y_all.shape, y_all.dtype)
print('test  images:', x_test_raw.shape, y_test_raw.shape)

print('\nclass distribution (full train / test):')
for i, name in enumerate(CLASS_NAMES):
    n_tr = int((y_all == i).sum())
    n_te = int((y_test_raw == i).sum())
    print(f'  {i} {name:12s} train={n_tr:5d} test={n_te:4d}')

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4.4))
for i, ax in enumerate(axes.ravel()):
    idx = np.where(y_all == i)[0][0]
    ax.imshow(x_all[idx], cmap='gray')
    ax.set_title(CLASS_NAMES[i], fontsize=9)
    ax.axis('off')
plt.suptitle('One sample per class (Fashion-MNIST)', fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'sec7_samples.png'), dpi=130)
plt.show()
plt.close()

تقسیم ۵۰٬۰۰۰ / ۱۰٬۰۰۰ با `stratify` تا نسبت کلاس‌ها حفظ شود، و نرمال‌سازی با
میانگین و انحراف‌معیارِ **فقط مجموعهٔ آموزش**.

In [ ]:
x_tr, x_va, y_tr, y_va = train_test_split(
    x_all, y_all, test_size=10_000, random_state=SEED, stratify=y_all)
print('train:', x_tr.shape[0], '| val:', x_va.shape[0])

mu = float(x_tr.mean() / 255.0)
sd = float(x_tr.std() / 255.0)
print(f'pixel mean={mu:.4f}  std={sd:.4f}  (min={x_tr.min()} max={x_tr.max()})')


def to_tensor(imgs):
    t = torch.from_numpy(imgs.astype(np.float32) / 255.0).unsqueeze(1)
    return (t - mu) / sd


def labels(ys):
    return torch.from_numpy(ys.astype(np.int64))


Xtr, ytr = to_tensor(x_tr), labels(y_tr)
Xva, yva = to_tensor(x_va), labels(y_va)
Xte, yte = to_tensor(x_test_raw), labels(y_test_raw)
print('tensor shapes:', Xtr.shape, Xva.shape, Xte.shape)

In [ ]:
BATCH = 128


def batches(X, y, bs=BATCH, shuffle=False):
    n = len(X)
    order = torch.randperm(n) if shuffle else torch.arange(n)
    for i in range(0, n, bs):
        idx = order[i:i + bs]
        yield X[idx], y[idx]


@torch.no_grad()
def evaluate(model, X, y, bs=1000):
    model.eval()
    preds = []
    for i in range(0, len(X), bs):
        logits = model(X[i:i + bs])
        preds.append(logits.argmax(1))
    p = torch.cat(preds).cpu().numpy()
    return accuracy_score(y.cpu().numpy(), p), p


def train_model(model, Xtr, ytr, Xva, yva, epochs, lr=1e-3, wd=1e-4,
                patience=None, tag='model'):
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    lossf = nn.CrossEntropyLoss()
    hist = {'epoch': [], 'train_loss': [], 'train_acc': [],
            'val_loss': [], 'val_acc': [], 'seconds': []}
    best_acc, best_state = -1.0, None
    t_start = time.perf_counter()

    for ep in range(1, epochs + 1):
        t0 = time.perf_counter()
        model.train()
        total, correct, running = 0, 0, 0.0
        for xb, yb in batches(Xtr, ytr, shuffle=True):
            opt.zero_grad()
            out = model(xb)
            loss = lossf(out, yb)
            loss.backward()
            opt.step()
            running += loss.item() * len(xb)
            correct += (out.argmax(1) == yb).sum().item()
            total += len(xb)
        val_acc, _ = evaluate(model, Xva, yva)
        val_loss = _val_loss(model, Xva, yva, lossf)
        dt = time.perf_counter() - t0
        hist['epoch'].append(ep)
        hist['train_loss'].append(running / total)
        hist['train_acc'].append(correct / total)
        hist['val_loss'].append(val_loss)
        hist['val_acc'].append(val_acc)
        hist['seconds'].append(round(dt, 2))
        print(f'  epoch {ep:2d}/{epochs}  train_loss={running / total:.4f} '
              f'acc={correct / total:.4f} | val_loss={val_loss:.4f} '
              f'val_acc={val_acc:.4f} | {dt:.1f}s')
        if val_acc > best_acc:
            best_acc = val_acc
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        if patience and ep - int(np.argmax(hist['val_acc'])) - 1 >= patience:
            print(f'  early stop at epoch {ep} (no val gain for {patience} epochs)')
            break
    hist['total_seconds'] = round(time.perf_counter() - t_start, 2)
    hist['best_val_acc'] = float(best_acc)
    model.load_state_dict(best_state)
    return hist


@torch.no_grad()
def _val_loss(model, X, y, lossf, bs=1000):
    model.eval()
    tot = 0.0
    for i in range(0, len(X), bs):
        tot += lossf(model(X[i:i + bs]), y[i:i + bs]).item() * len(X[i:i + bs])
    return tot / len(X)

## ۷.۲ MLP با دو لایهٔ پنهان

`784 → 512 → 256 → 10`، تصاویر تخت‌شده، فعال‌سازی ReLU.

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 512), nn.ReLU(),
            nn.Linear(512, 256), nn.ReLU(),
            nn.Linear(256, 10),
        )

    def forward(self, x):
        return self.net(x)


mlp = MLP()
n_params_mlp = sum(p.numel() for p in mlp.parameters())
print('MLP parameters:', f'{n_params_mlp:,}')
print(mlp)

In [ ]:
print('\ntraining MLP:')
hist_mlp = train_model(mlp, Xtr, ytr, Xva, yva, epochs=12, lr=1e-3, wd=1e-4, tag='mlp')
print(f'MLP best val acc = {hist_mlp["best_val_acc"]:.4f} '
      f'(train time {hist_mlp["total_seconds"]}s)')

## ۷.۳ CNN با دو بلوک کانولوشن/Pooling

هر بلوک: `Conv3×3 → BatchNorm → ReLU → MaxPool2`. کنترل بیش‌برازش: **Dropout**
(۰٫۴ و ۰٫۳) + **weight decay** + **early stopping** روی اعتبارسنجی.

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.4),
            nn.Linear(64 * 7 * 7, 128), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


cnn = CNN()
n_params_cnn = sum(p.numel() for p in cnn.parameters())
print('CNN parameters:', f'{n_params_cnn:,}')
print(cnn)

In [ ]:
print('\ntraining CNN:')
hist_cnn = train_model(cnn, Xtr, ytr, Xva, yva, epochs=8, lr=1e-3, wd=1e-4,
                       patience=2, tag='cnn')
print(f'CNN best val acc = {hist_cnn["best_val_acc"]:.4f} '
      f'(train time {hist_cnn["total_seconds"]}s)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.9))
for name, h, c in [('MLP', hist_mlp, '#3b6ea5'), ('CNN', hist_cnn, '#c0392b')]:
    axes[0].plot(h['epoch'], h['train_loss'], '--', color=c, alpha=.55, label=f'{name} train')
    axes[0].plot(h['epoch'], h['val_loss'], '-o', color=c, ms=3, label=f'{name} val')
    axes[1].plot(h['epoch'], h['train_acc'], '--', color=c, alpha=.55, label=f'{name} train')
    axes[1].plot(h['epoch'], h['val_acc'], '-o', color=c, ms=3, label=f'{name} val')
axes[0].set(xlabel='epoch', ylabel='cross-entropy', title='Loss curves')
axes[1].set(xlabel='epoch', ylabel='accuracy', title='Accuracy curves')
for ax in axes:
    ax.grid(alpha=.3)
    ax.legend(fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'sec7_curves.png'), dpi=130)
plt.show()
plt.close()
print('curves saved')

## ۷.۴ ارزیابی نهایی روی آزمون (فقط یک بار)

In [ ]:
models = {'MLP': (mlp, hist_mlp), 'CNN': (cnn, hist_cnn)}
best_name = max(models, key=lambda k: models[k][1]['best_val_acc'])
best_model = models[best_name][0]
print('selected model:', best_name)

test_acc, test_pred = evaluate(best_model, Xte, yte)
y_true = yte.cpu().numpy()
macro_f1 = f1_score(y_true, test_pred, average='macro')
print(f'TEST accuracy={test_acc:.4f}  macro-F1={macro_f1:.4f}')

cm = confusion_matrix(y_true, test_pred)
cm_norm = cm / cm.sum(axis=1, keepdims=True)
print('\nper-class recall:')
for i, name in enumerate(CLASS_NAMES):
    print(f'  {name:12s} {cm_norm[i, i]:.3f}  (n={cm[i].sum()})')

In [ ]:
fig, ax = plt.subplots(figsize=(7.6, 6.6))
im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(10)); ax.set_yticks(range(10))
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(CLASS_NAMES, fontsize=8)
for i in range(10):
    for j in range(10):
        if cm_norm[i, j] > 0.01:
            ax.text(j, i, f'{cm_norm[i, j]:.2f}', ha='center', va='center', fontsize=6,
                    color='white' if cm_norm[i, j] > .55 else 'black')
ax.set(xlabel='predicted', ylabel='true', title=f'Confusion matrix (row-normalised) — {best_name}')
plt.colorbar(im, ax=ax, fraction=.046)
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'sec7_confusion.png'), dpi=130)
plt.show()
plt.close()

off = cm.copy()
np.fill_diagonal(off, 0)
pairs = []
for i in range(10):
    for j in range(i + 1, 10):
        pairs.append((off[i, j] + off[j, i], i, j))
pairs.sort(reverse=True)
print('\ntop confused pairs:')
for cnt, i, j in pairs[:5]:
    print(f'  {CLASS_NAMES[i]} <-> {CLASS_NAMES[j]}: {cnt} errors '
          f'({off[i, j]} as {CLASS_NAMES[j]}, {off[j, i]} as {CLASS_NAMES[i]})')

In [ ]:
wrong = np.where(test_pred != y_true)[0][:12]
fig, axes = plt.subplots(3, 4, figsize=(9, 7))
for ax, k in zip(axes.ravel(), wrong):
    ax.imshow(x_test_raw[k], cmap='gray')
    ax.set_title(f'true: {CLASS_NAMES[y_true[k]]}\npred: {CLASS_NAMES[test_pred[k]]}', fontsize=7)
    ax.axis('off')
plt.suptitle(f'Misclassified test images — {best_name}', fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'sec7_errors.png'), dpi=130)
plt.show()
plt.close()
print('error grid saved')

In [ ]:
results = {
    'n_train_full': int(len(x_all)),
    'n_test': int(len(x_test_raw)),
    'n_train': int(len(x_tr)),
    'n_val': int(len(x_va)),
    'pixel_mean': round(mu, 4),
    'pixel_std': round(sd, 4),
    'class_counts_train': {CLASS_NAMES[i]: int((y_all == i).sum()) for i in range(10)},
    'class_counts_test': {CLASS_NAMES[i]: int((y_test_raw == i).sum()) for i in range(10)},
    'mlp': {'params': int(n_params_mlp), 'best_val_acc': hist_mlp['best_val_acc'],
            'train_seconds': hist_mlp['total_seconds'],
            'epochs_run': len(hist_mlp['epoch'])},
    'cnn': {'params': int(n_params_cnn), 'best_val_acc': hist_cnn['best_val_acc'],
            'train_seconds': hist_cnn['total_seconds'],
            'epochs_run': len(hist_cnn['epoch']),
            'seconds_per_epoch': round(float(np.mean(hist_cnn['seconds'])), 2)},
    'selected': best_name,
    'test': {'accuracy': float(test_acc), 'macro_f1': float(macro_f1)},
    'per_class_recall': {CLASS_NAMES[i]: round(float(cm_norm[i, i]), 3) for i in range(10)},
    'top_pairs': [{'pair': f'{CLASS_NAMES[i]} <-> {CLASS_NAMES[j]}', 'errors': int(c),
                   'i_as_j': int(off[i, j]), 'j_as_i': int(off[j, i])}
                  for c, i, j in pairs[:5]],
}
with open(os.path.join(OUT, 'sec7_results.json'), 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print('saved out/sec7_results.json')

---

## بخش ۸ — پردازش زبان و تشخیص پیامک اسپم

طبقه‌بندی متن پیامک به `ham` یا `spam`. نکتهٔ کلیدی: ۴۰۳ ردیف متن دقیقاً تکراری‌اند،
پس تقسیم باید **گروهی** باشد تا نسخهٔ تکراری در دو بخش نیفتد و اعتبارسنجی متورم نشود.

In [ ]:
import json
import os
import re
import time
import warnings

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

warnings.filterwarnings('ignore')
SEED = 42
DATA = '../Day2_Datasets/04_SmsSpam/sms_spam.csv'
OUT = 'out'
os.makedirs(OUT, exist_ok=True)

## ۸.۱ شناخت داده و تقسیم گروهی

In [ ]:
df = pd.read_csv(DATA)
df['label'] = df['label'].str.strip().str.lower()
df['y'] = (df['label'] == 'spam').astype(int)

print('rows:', len(df), '| spam: %d (%.2f%%)' % (df.y.sum(), 100 * df.y.mean()))
print('unique texts:', df.text.nunique())
dup_rows = int(df.text.duplicated().sum())
print('exactly duplicated rows (beyond first copy):', dup_rows)

lengths = df.text.str.len()
words = df.text.str.split().str.len()
print('\nchar length  : mean %.1f | median %.0f | p95 %.0f | max %d'
      % (lengths.mean(), lengths.median(), lengths.quantile(.95), lengths.max()))
print('word length  : mean %.1f | median %.0f | p95 %.0f | max %d'
      % (words.mean(), words.median(), words.quantile(.95), words.max()))
print('char length by class:')
print(df.assign(chars=lengths, words=words).groupby('label')[['chars', 'words']]
      .agg(['mean', 'median']).round(1).to_string())

### تقسیم گروهی: هر متنِ یکسان کامل به یک بخش می‌رود

ابتدا متن‌های یکتا استخراج می‌شوند (۵٬۱۷۱ متن از ۵٬۵۷۴ ردیف)، سپس **گروه‌ها** با
`stratify` به ۷۰/۱۵/۱۵ تقسیم می‌شوند و برچسب هر ردیف از گروهش می‌آید. این کار
جلوگیری می‌کند همان پیامک هم در آموزش دیده شود و هم در اعتبارسنجی/آزمون نمره بگیرد.

In [ ]:
group = (df.groupby('text')
           .agg(y=('y', 'max'), n=('y', 'size'))
           .reset_index())
print('unique groups:', len(group), '| groups with >1 row:', int((group.n > 1).sum()))

g_tr, g_rest = train_test_split(group, test_size=0.30, random_state=SEED,
                                stratify=group['y'])
g_va, g_te = train_test_split(g_rest, test_size=0.50, random_state=SEED,
                              stratify=g_rest['y'])

split_map = {}
for name, g in [('train', g_tr), ('val', g_va), ('test', g_te)]:
    for t in g['text']:
        split_map[t] = name
df['split'] = df['text'].map(split_map)

print('\nsplit sizes (rows):')
for s in ['train', 'val', 'test']:
    sub = df[df.split == s]
    print(f'  {s:5s} rows={len(sub):5d} spam={int(sub.y.sum()):4d} '
          f'({100 * sub.y.mean():5.2f}%) | unique texts={sub.text.nunique()}')

overlap = (set(df[df.split == 'train'].text) & set(df[df.split == 'val'].text))
overlap2 = (set(df[df.split == 'train'].text) & set(df[df.split == 'test'].text))
print('text overlap train/val:', len(overlap), '| train/test:', len(overlap2))

X_tr = df.loc[df.split == 'train', 'text']
y_tr = df.loc[df.split == 'train', 'y']
X_va = df.loc[df.split == 'val', 'text']
y_va = df.loc[df.split == 'val', 'y']
X_te = df.loc[df.split == 'test', 'text']
y_te = df.loc[df.split == 'test', 'y']

## ۸.۲ خط پایه: TF-IDF + LogisticRegression

همهٔ مراحل داخل `Pipeline` است؛ `TfidfVectorizer` فقط روی `fit(X_train)` واژگان و
IDF را می‌آموزد، پس val/test هیچ سهمی در ساخت واژگان ندارند.

In [ ]:
def make_tfidf(max_features=None):
    return TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=2,
        max_features=max_features,
        sublinear_tf=True,
        strip_accents='unicode',
    )


def make_lr():
    return LogisticRegression(max_iter=2000, C=1.0, solver='liblinear',
                              random_state=SEED)


baseline = Pipeline([('tfidf', make_tfidf()), ('clf', make_lr())])
t0 = time.perf_counter()
baseline.fit(X_tr, y_tr)
lr_time = time.perf_counter() - t0

p_va = baseline.predict_proba(X_va)[:, 1]
pred_va = (p_va >= 0.5).astype(int)
b = {
    'precision': float(precision_score(y_va, pred_va)),
    'recall': float(recall_score(y_va, pred_va)),
    'macro_f1': float(f1_score(y_va, pred_va, average='macro')),
    'pr_auc': float(average_precision_score(y_va, p_va)),
    'fit_seconds': round(lr_time, 3),
}
print('LR baseline (val):', json.dumps(b, indent=2))
print('vocabulary size:', len(baseline.named_steps['tfidf'].vocabulary_))

## ۸.۳ مقایسهٔ MultinomialNB و LinearSVC با خط پایه

تنظیم پارامتر فقط روی آموزش/اعتبارسنجی: `alpha` برای NB و `C` برای LinearSVC
(و برای LR) روی اعتبارسنجی جست‌وجو می‌شود.

In [ ]:
def scores(y_true, score_or_proba):
    """PR-AUC و پیش‌بینی‌های آستانهٔ ۰٫۵ را برمی‌گرداند."""
    return score_or_proba


def evaluate_model(name, pipe, X, y, proba, fit_seconds, params):
    pred = (proba >= 0.5).astype(int)
    return {
        'name': name,
        'precision': float(precision_score(y, pred)),
        'recall': float(recall_score(y, pred)),
        'macro_f1': float(f1_score(y, pred, average='macro')),
        'pr_auc': float(average_precision_score(y, proba)),
        'fit_seconds': round(fit_seconds, 3),
        'params': params,
    }


results = {}
# --- LR با تنظیم C
best_lr = None
for C in [0.3, 1.0, 3.0]:
    pipe = Pipeline([('tfidf', make_tfidf()), ('clf', make_lr().set_params(C=C))])
    t0 = time.perf_counter()
    pipe.fit(X_tr, y_tr)
    dt = time.perf_counter() - t0
    pr = pipe.predict_proba(X_va)[:, 1]
    ma = f1_score(y_va, (pr >= .5).astype(int), average='macro')
    print(f'LR C={C}: val macro-F1={ma:.4f} PR-AUC={average_precision_score(y_va, pr):.4f}')
    if best_lr is None or ma > best_lr[0]:
        best_lr = (ma, C, pipe, dt, pr)
_, C_lr, lr_pipe, lr_dt, lr_pr = best_lr
results['logreg'] = evaluate_model('LogisticRegression', lr_pipe, X_va, y_va,
                                   lr_pr, lr_dt, f'C={C_lr}')

# --- MultinomialNB با تنظیم alpha
best_nb = None
for alpha in [0.1, 0.3, 1.0]:
    pipe = Pipeline([('tfidf', make_tfidf()),
                     ('clf', MultinomialNB(alpha=alpha))])
    t0 = time.perf_counter()
    pipe.fit(X_tr, y_tr)
    dt = time.perf_counter() - t0
    pr = pipe.predict_proba(X_va)[:, 1]
    ma = f1_score(y_va, (pr >= .5).astype(int), average='macro')
    print(f'NB alpha={alpha}: val macro-F1={ma:.4f} PR-AUC={average_precision_score(y_va, pr):.4f}')
    if best_nb is None or ma > best_nb[0]:
        best_nb = (ma, alpha, pipe, dt, pr)
_, alpha_nb, nb_pipe, nb_dt, nb_pr = best_nb
results['naive_bayes'] = evaluate_model('MultinomialNB', nb_pipe, X_va, y_va,
                                        nb_pr, nb_dt, f'alpha={alpha_nb}')

# --- LinearSVC با تنظیم C
best_svc = None
for C in [0.1, 1.0, 3.0]:
    pipe = Pipeline([('tfidf', make_tfidf()), ('clf', LinearSVC(C=C, random_state=SEED))])
    t0 = time.perf_counter()
    pipe.fit(X_tr, y_tr)
    dt = time.perf_counter() - t0
    dec = pipe.decision_function(X_va)
    pred = (dec >= 0).astype(int)
    ma = f1_score(y_va, pred, average='macro')
    print(f'LinearSVC C={C}: val macro-F1={ma:.4f} PR-AUC={average_precision_score(y_va, dec):.4f}')
    if best_svc is None or ma > best_svc[0]:
        best_svc = (ma, C, pipe, dt, dec)
_, C_svc, svc_pipe, svc_dt, svc_dec = best_svc
results['linear_svc'] = evaluate_model('LinearSVC', svc_pipe, X_va, y_va,
                                       svc_dec, svc_dt, f'C={C_svc}')

print('\nvalidation summary:')
for k, v in results.items():
    print(f"  {v['name']:20s} P={v['precision']:.4f} R={v['recall']:.4f} "
          f"macroF1={v['macro_f1']:.4f} PR-AUC={v['pr_auc']:.4f} "
          f"fit={v['fit_seconds']}s ({v['params']})")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.6))
for name, pr, color in [('LogisticRegression', lr_pr, '#3b6ea5'),
                        ('MultinomialNB', nb_pr, '#c0392b'),
                        ('LinearSVC (decision)', svc_dec, '#2e7d32')]:
    p, r, _ = precision_recall_curve(y_va, pr)
    ax.plot(r, p, label=f'{name} (AP={average_precision_score(y_va, pr):.3f})',
            color=color)
ax.axhline(y_va.mean(), ls='--', c='grey', label=f'prevalence {y_va.mean():.2f}')
ax.set(xlabel='spam recall', ylabel='spam precision', title='PR curves — validation')
ax.grid(alpha=.3)
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'sec8_pr.png'), dpi=130)
plt.show()
plt.close()

## ۸.۴ آستانه با Precision ≥ ۰٫۹۰ برای مدل احتمالاتی

In [ ]:
prob_name = 'naive_bayes' if results['naive_bayes']['pr_auc'] >= results['logreg']['pr_auc'] else 'logreg'
prob_pipe = nb_pipe if prob_name == 'naive_bayes' else lr_pipe
pv = nb_pr if prob_name == 'naive_bayes' else lr_pr
print('probabilistic model used for thresholding:', results[prob_name]['name'])

rows = []
for thr in np.round(np.arange(0.05, 0.951, 0.01), 2):
    pred = (pv >= thr).astype(int)
    rows.append((thr, precision_score(y_va, pred), recall_score(y_va, pred)))
sweep = pd.DataFrame(rows, columns=['thr', 'precision', 'recall'])
feasible = sweep[sweep.precision >= 0.90]
if len(feasible):
    best_row = feasible.sort_values(['recall', 'precision'], ascending=False).iloc[0]
else:
    best_row = sweep.sort_values('precision', ascending=False).iloc[0]
thr_best = float(best_row.thr)
print(f'chosen threshold={thr_best:.2f} -> val precision={best_row.precision:.4f} '
      f'recall={best_row.recall:.4f}')

fig, ax = plt.subplots(figsize=(7.4, 4.4))
ax.plot(sweep.thr, sweep.precision, label='precision', color='#3b6ea5')
ax.plot(sweep.thr, sweep.recall, label='recall', color='#c0392b')
ax.axhline(0.90, ls=':', c='k', lw=1, label='precision target 0.90')
ax.axvline(thr_best, ls='--', c='g', label=f'thr={thr_best:.2f}')
ax.set(xlabel='threshold', ylabel='score', title='Threshold sweep on validation')
ax.grid(alpha=.3)
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'sec8_threshold.png'), dpi=130)
plt.show()
plt.close()

In [ ]:
pte = prob_pipe.predict_proba(X_te)[:, 1]
pred_te = (pte >= thr_best).astype(int)
test_metrics = {
    'threshold': thr_best,
    'precision': float(precision_score(y_te, pred_te)),
    'recall': float(recall_score(y_te, pred_te)),
    'macro_f1': float(f1_score(y_te, pred_te, average='macro')),
    'pr_auc': float(average_precision_score(y_te, pte)),
    'accuracy': float((pred_te == y_te.to_numpy()).mean()),
}
print('TEST:', json.dumps(test_metrics, indent=2))

cm = confusion_matrix(y_te, pred_te)
print('\nconfusion matrix [ham, spam]:\n', cm)

fig, ax = plt.subplots(figsize=(4.2, 3.6))
ax.imshow(cm, cmap='Blues')
for i in range(2):
    for j in range(2):
        ax.text(j, i, f'{cm[i, j]}', ha='center', va='center',
                color='white' if cm[i, j] > cm.max() / 2 else 'black')
ax.set(xticks=[0, 1], yticks=[0, 1], xlabel='predicted', ylabel='true',
       title=f'Test @ thr={thr_best:.2f}')
ax.set_xticklabels(['ham', 'spam'])
ax.set_yticklabels(['ham', 'spam'])
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'sec8_confusion.png'), dpi=130)
plt.show()
plt.close()

In [ ]:
test_df = pd.DataFrame({'text': X_te.to_numpy(), 'y': y_te.to_numpy(),
                        'p': pte, 'pred': pred_te})
fp = (test_df[(test_df.pred == 1) & (test_df.y == 0)]
      .sort_values('p', ascending=False).drop_duplicates('text'))
fn = (test_df[(test_df.pred == 0) & (test_df.y == 1)]
      .sort_values('p', ascending=False).drop_duplicates('text'))
print(f'false positives: {len(fp)} | false negatives: {len(fn)}')

errors = []
for _, r in fp.head(3).iterrows():
    errors.append({'type': 'FP', 'p': round(float(r.p), 3),
                   'text': r.text.strip()[:220]})
for _, r in fn.head(2).iterrows():
    errors.append({'type': 'FN', 'p': round(float(r.p), 3),
                   'text': r.text.strip()[:220]})
for e in errors:
    print(f"\n[{e['type']}] p={e['p']}\n  {e['text'][:160]}")

In [ ]:
res = {
    'n_rows': int(len(df)),
    'n_unique_texts': int(df.text.nunique()),
    'n_duplicate_rows': dup_rows,
    'spam_pct': round(100 * float(df.y.mean()), 2),
    'length_chars': {'mean': round(float(lengths.mean()), 1),
                     'median': float(lengths.median()),
                     'p95': float(lengths.quantile(.95)),
                     'max': int(lengths.max())},
    'length_words': {'mean': round(float(words.mean()), 1),
                     'median': float(words.median()),
                     'max': int(words.max())},
    'split': {s: {'rows': int((df.split == s).sum()),
                  'spam': int(df.loc[df.split == s, 'y'].sum()),
                  'spam_pct': round(100 * float(df.loc[df.split == s, 'y'].mean()), 2)}
              for s in ['train', 'val', 'test']},
    'overlap_train_val': len(overlap),
    'overlap_train_test': len(overlap2),
    'baseline_val': b,
    'vocab_size': int(len(baseline.named_steps['tfidf'].vocabulary_)),
    'models_val': results,
    'probabilistic_model': results[prob_name]['name'],
    'threshold': thr_best,
    'threshold_val': {'precision': float(best_row.precision),
                      'recall': float(best_row.recall)},
    'test': test_metrics,
    'confusion': cm.tolist(),
    'errors': errors,
}
with open(os.path.join(OUT, 'sec8_results.json'), 'w', encoding='utf-8') as f:
    json.dump(res, f, indent=2, ensure_ascii=False)
print('\nsaved out/sec8_results.json')

---

## بخش ۹ — پردازش صوت و تعمیم به گویندهٔ جدید (Spoken Digits)

تشخیص رقم گفتاری از فایل‌های `{digit}_{speaker}_{index}.wav`. دو طرح ارزیابی:
۱) **تقسیم تصادفی فایل** (همهٔ گویندگان در آموزش و آزمون) ۲) **تفکیک بر اساس گوینده**
(دو گویندهٔ کاملاً دیده‌نشده). تفاوت این دو، همان شکاف تعمیم است.

In [ ]:
import glob
import json
import os
import time
import warnings
from collections import Counter

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from scipy.fft import dct
from scipy.io import wavfile
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix, f1_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA = '../Day2_Datasets/05_SpokenDigits/recordings'
OUT = 'out'
os.makedirs(OUT, exist_ok=True)

## ۹.۱ بررسی داده: نرخ نمونه‌برداری، طول و شمارش

In [ ]:
paths = sorted(glob.glob(os.path.join(DATA, '*.wav')))
print('files:', len(paths))

meta = []
waves = {}
for p in paths:
    name = os.path.basename(p)[:-4]
    digit, speaker, idx = name.split('_')
    sr, x = wavfile.read(p)
    x = x.astype(np.float32) / 32768.0
    waves[name] = x
    meta.append({'name': name, 'digit': int(digit), 'speaker': speaker,
                 'sr': sr, 'n': len(x), 'dur': len(x) / sr})

print('sample rates:', Counter(m['sr'] for m in meta))
durs = np.array([m['dur'] for m in meta])
print('duration: mean %.3f s | min %.3f | max %.3f | median %.3f'
      % (durs.mean(), durs.min(), durs.max(), np.median(durs)))
print('per digit :', dict(sorted(Counter(m['digit'] for m in meta).items())))
print('per speaker:', dict(Counter(m['speaker'] for m in meta)))
print('per speaker x digit (matrix):')
speakers = sorted({m['speaker'] for m in meta})
mat = np.zeros((len(speakers), 10), dtype=int)
for m in meta:
    mat[speakers.index(m['speaker']), m['digit']] += 1
print('        ' + ' '.join(f'{d:>4d}' for d in range(10)))
for s, row in zip(speakers, mat):
    print(f'{s:8s}' + ' '.join(f'{v:>4d}' for v in row))

### استخراج Log-Mel و MFCC (پیاده‌سازی با NumPy/SciPy، بدون کتابخانهٔ اضافی)

In [ ]:
SR = 8000
FRAME, HOP, NFFT, N_MELS, N_MFCC = 256, 80, 256, 40, 13


def mel_filterbank(sr=SR, n_fft=NFFT, n_mels=N_MELS, fmin=20.0, fmax=None):
    fmax = fmax or sr / 2
    hz2mel = lambda f: 2595 * np.log10(1 + f / 700)
    mel2hz = lambda m: 700 * (10 ** (m / 2595) - 1)
    mels = np.linspace(hz2mel(fmin), hz2mel(fmax), n_mels + 2)
    hz = mel2hz(mels)
    bins = np.floor((n_fft + 1) * hz / sr).astype(int)
    fb = np.zeros((n_mels, n_fft // 2 + 1))
    for m in range(1, n_mels + 1):
        lo, ce, hi = bins[m - 1], bins[m], bins[m + 1]
        if ce == lo:
            ce = lo + 1
        if hi == ce:
            hi = ce + 1
        for k in range(lo, ce):
            fb[m - 1, k] = (k - lo) / (ce - lo)
        for k in range(ce, hi):
            fb[m - 1, k] = (hi - k) / (hi - ce)
    return fb


FB = mel_filterbank()
DCT = dct(np.eye(N_MELS), type=2, norm='ortho')[:N_MFCC]


def logmel(x, sr=SR):
    """log-Mel spectrogram: (n_mels, frames)"""
    if len(x) < FRAME:
        x = np.pad(x, (0, FRAME - len(x)))
    x = np.append(x[0], x[1:] - 0.97 * x[:-1])          # pre-emphasis
    n_frames = 1 + (len(x) - FRAME) // HOP
    idx = np.arange(FRAME)[None, :] + HOP * np.arange(n_frames)[:, None]
    frames = x[idx] * np.hamming(FRAME)[None, :]
    spec = np.abs(np.fft.rfft(frames, NFFT)) ** 2
    mel = spec @ FB.T
    return (10 * np.log10(mel + 1e-10)).T      # (n_mels, n_frames)


def mfcc_from_logmel(L):
    return DCT @ L


def delta(feat, N=2):
    pad = np.pad(feat, ((0, 0), (N, N)), mode='edge')
    denom = 2 * sum(n ** 2 for n in range(1, N + 1))
    out = np.zeros_like(feat)
    for n in range(1, N + 1):
        out += n * (pad[:, N + n:N + n + feat.shape[1]] - pad[:, N - n:N - n + feat.shape[1]])
    return out / denom


def summary_features(x, drop_c0=False):
    L = logmel(x)
    M = mfcc_from_logmel(L)
    dM = delta(M)
    if drop_c0:                      # حذف ضریب صفرم (لگاریتم انرژی) که وابسته به گوینده است
        M, dM = M[1:], dM[1:]
    spec = np.abs(np.fft.rfft(x * np.hamming(len(x)))) ** 2
    freqs = np.fft.rfftfreq(len(x), 1 / SR)
    p = spec / (spec.sum() + 1e-12)
    centroid = float((freqs * p).sum())
    bandwidth = float(np.sqrt(((freqs - centroid) ** 2 * p).sum()))
    csum = np.cumsum(p)
    rolloff = float(freqs[np.searchsorted(csum, 0.85)])
    zcr = float(np.mean(np.abs(np.diff(np.sign(x))) > 0))
    return np.concatenate([
        M.mean(1), M.std(1), dM.mean(1), dM.std(1),
        [centroid, bandwidth, rolloff, zcr, len(x) / SR],
    ])

### رسم موج و Log-Mel چند نمونه

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(12, 4.6))
samples = [('0', 'george'), ('3', 'jackson'), ('7', 'lucas'), ('9', 'yweweler')]
for k, (d, s) in enumerate(samples):
    name = f'{d}_{s}_0'
    x = waves[name]
    t = np.arange(len(x)) / SR
    axes[0, k].plot(t, x, lw=.8, color='#3b6ea5')
    axes[0, k].set_title(f'digit {d} — {s}', fontsize=9)
    axes[0, k].set_xlabel('time (s)', fontsize=8)
    if k == 0:
        axes[0, k].set_ylabel('waveform', fontsize=9)
    L = logmel(x)
    im = axes[1, k].imshow(L, aspect='auto', origin='lower', cmap='magma')
    axes[1, k].set_title('log-Mel', fontsize=9)
    axes[1, k].set_xlabel('frame', fontsize=8)
    if k == 0:
        axes[1, k].set_ylabel('mel bin', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'sec9_wave_mel.png'), dpi=130)
plt.show()
plt.close()
print('waveform/spectrogram figure saved')

## دو طرح تقسیم

| طرح | آموزش | اعتبارسنجی | آزمون | نکته |
|---|---|---|---|---|
| تصادفی فایل | ۷۰٪ فایل‌ها | ۱۵٪ | ۱۵٪ | همهٔ گویندگان در هر سه بخش |
| تفکیک گوینده | ۴ گوینده | `theo` | `yweweler` | صفر اشتراک گوینده بین بخش‌ها |

In [ ]:
import pandas as pd

df = pd.DataFrame(meta)
rng = np.random.RandomState(SEED)
idx = rng.permutation(len(df))
a_tr, a_rest = idx[:int(.70 * len(df))], idx[int(.70 * len(df)):]
a_va, a_te = a_rest[:len(a_rest) // 2], a_rest[len(a_rest) // 2:]
random_split = {'train': df.iloc[a_tr], 'val': df.iloc[a_va], 'test': df.iloc[a_te]}

held = ['theo', 'yweweler']
speaker_split = {
    'train': df[~df.speaker.isin(held)],
    'val': df[df.speaker == 'theo'],
    'test': df[df.speaker == 'yweweler'],
}

for name, sp in [('random', random_split), ('speaker', speaker_split)]:
    print(f'\n{name} split:')
    for part in ['train', 'val', 'test']:
        s = sp[part]
        print(f'  {part:5s} n={len(s):4d} speakers={sorted(s.speaker.unique())}')

## ۹.۲ ویژگی‌های MFCC + مدل کلاسیک (SVM)

برای هر فایل بردار ۶۰بعدی ساخته می‌شود: میانگین و انحراف‌معیارِ MFCC (۱۳+۱۳)،
میانگین و انحراف‌معیارِ دلتا-MFCC (۱۳+۱۳)، و پنج ویژگی طیفی/زمانی (مرکز طیف،
پهنای باند، roll-off ۸۵٪، نرخ عبور از صفر، طول). استانداردسازی و انتخاب پارامتر
داخل `Pipeline` و `GridSearchCV` **فقط روی آموزش** انجام می‌شود.

In [ ]:
t0 = time.perf_counter()
F = {name: summary_features(x) for name, x in waves.items()}
print('feature extraction: %.1fs for %d files' % (time.perf_counter() - t0, len(F)))


def build_X(split):
    def mat(part):
        rows = split[part]
        return np.vstack([F[n] for n in rows['name']]), rows['digit'].to_numpy()
    return mat('train'), mat('val'), mat('test')


svm_grid = {'clf__C': [1, 10, 100], 'clf__gamma': ['scale', 0.01]}

svm_results = {}
for scheme in ['random', 'speaker']:
    (Xtr, ytr), (Xva, yva), (Xte, yte) = build_X(
        random_split if scheme == 'random' else speaker_split)
    pipe = Pipeline([('scale', StandardScaler()),
                     ('clf', SVC(kernel='rbf', random_state=SEED))])
    gs = GridSearchCV(pipe, svm_grid, cv=3, scoring='f1_macro', n_jobs=1)
    t0 = time.perf_counter()
    gs.fit(Xtr, ytr)
    dt = time.perf_counter() - t0
    pred_va = gs.predict(Xva)
    pred_te = gs.predict(Xte)
    svm_results[scheme] = {
        'best_params': {k.replace('clf__', ''): str(v) for k, v in gs.best_params_.items()},
        'cv_macro_f1': float(gs.best_score_),
        'val_macro_f1': float(f1_score(yva, pred_va, average='macro')),
        'test_macro_f1': float(f1_score(yte, pred_te, average='macro')),
        'fit_seconds': round(dt, 2),
        'val_pred': pred_va, 'val_true': yva,
        'test_pred': pred_te, 'test_true': yte,
    }
    print(f"SVM [{scheme:7s}] best={gs.best_params_} CV={gs.best_score_:.4f} "
          f"VAL={svm_results[scheme]['val_macro_f1']:.4f} "
          f"TEST={svm_results[scheme]['test_macro_f1']:.4f} ({dt:.1f}s)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
for ax, scheme in zip(axes, ['random', 'speaker']):
    cm = confusion_matrix(svm_results[scheme]['val_true'],
                          svm_results[scheme]['val_pred'], labels=range(10))
    im = ax.imshow(cm, cmap='Blues')
    ax.set_title(f'SVM confusion — {scheme} split (val)', fontsize=9)
    ax.set_xticks(range(10)); ax.set_yticks(range(10))
    ax.set_xlabel('predicted'); ax.set_ylabel('true')
    for i in range(10):
        for j in range(10):
            if cm[i, j]:
                ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=6,
                        color='white' if cm[i, j] > cm.max() / 2 else 'black')
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'sec9_svm_confusion.png'), dpi=130)
plt.show()
plt.close()

## ۹.۳ شبکهٔ کانولوشنی کوچک روی Log-Mel

**طول ورودی:** یک ثانیه (۸٬۰۰۰ نمونه) با **zero-padding** یا **truncation**؛ چون
بلندترین فایل ۱٫۱۶۸ ثانیه و میانه ۰٫۴۹ ثانیه است. Log-Mel با ۴۰ فیلتر مِل و گام
۱۰ میلی‌ثانیه → ورودی ثابت **۱ × ۴۰ × ۱۰۱**.

In [ ]:
MAX_LEN = SR  # 1 second
N_FRAMES = 1 + (MAX_LEN - FRAME) // HOP


def pad_or_truncate(x, n=MAX_LEN):
    if len(x) >= n:
        return x[:n]
    return np.pad(x, (0, n - len(x)))


def spec_tensor(x):
    L = logmel(pad_or_truncate(x))
    L = L[:, :N_FRAMES]
    if L.shape[1] < N_FRAMES:
        L = np.pad(L, ((0, 0), (0, N_FRAMES - L.shape[1])), mode='edge')
    L = (L - L.mean()) / (L.std() + 1e-6)
    return torch.tensor(L[None, :, :], dtype=torch.float32)


print('CNN input shape:', spec_tensor(waves['0_george_0']).shape)

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
        )
        flat = 32 * (N_MELS // 4) * (N_FRAMES // 4)   # دو بار MaxPool(2)
        self.head = nn.Sequential(
            nn.Flatten(), nn.Dropout(0.3),
            nn.Linear(flat, 64), nn.ReLU(),
            nn.Dropout(0.2), nn.Linear(64, n_classes),
        )

    def forward(self, x):
        return self.head(self.features(x))


def train_cnn(split, epochs=14, bs=64):
    tr, va = split['train'], split['val']
    Xtr = torch.stack([spec_tensor(waves[n]) for n in tr['name']])
    ytr = torch.tensor(tr['digit'].to_numpy())
    Xva = torch.stack([spec_tensor(waves[n]) for n in va['name']])
    yva = torch.tensor(va['digit'].to_numpy())

    model = SmallCNN()
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    lossf = nn.CrossEntropyLoss()
    hist = {'epoch': [], 'train_loss': [], 'val_macro_f1': []}
    best_f1, best_state = -1, None
    t0 = time.perf_counter()
    for ep in range(1, epochs + 1):
        model.train()
        perm = torch.randperm(len(Xtr))
        tot = 0.0
        for i in range(0, len(Xtr), bs):
            idx = perm[i:i + bs]
            opt.zero_grad()
            loss = lossf(model(Xtr[idx]), ytr[idx])
            loss.backward()
            opt.step()
            tot += loss.item() * len(idx)
        model.eval()
        with torch.no_grad():
            pv = model(Xva).argmax(1).numpy()
        f1 = f1_score(yva.numpy(), pv, average='macro')
        hist['epoch'].append(ep)
        hist['train_loss'].append(tot / len(Xtr))
        hist['val_macro_f1'].append(float(f1))
        if f1 > best_f1:
            best_f1 = f1
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    hist['train_seconds'] = round(time.perf_counter() - t0, 1)
    return model, hist, best_f1


def eval_cnn(model, part):
    X = torch.stack([spec_tensor(waves[n]) for n in part['name']])
    with torch.no_grad():
        pred = model(X).argmax(1).numpy()
    y = part['digit'].to_numpy()
    return y, pred, float(f1_score(y, pred, average='macro'))


cnn_results = {}
for scheme in ['random', 'speaker']:
    split = random_split if scheme == 'random' else speaker_split
    print(f'\ntraining CNN [{scheme}] ...')
    model, hist, best_val = train_cnn(split)
    y_te, p_te, f1_te = eval_cnn(model, split['test'])
    cnn_results[scheme] = {'val_macro_f1': float(best_val), 'test_macro_f1': f1_te,
                           'train_seconds': hist['train_seconds'],
                           'hist': hist, 'model': model,
                           'test_true': y_te, 'test_pred': p_te}
    print(f'  CNN [{scheme:7s}] VAL={best_val:.4f} TEST={f1_te:.4f} '
          f'({hist["train_seconds"]}s)')
    cnn_results[scheme]['model'] = model

In [ ]:
fig, ax = plt.subplots(figsize=(7.4, 4.2))
for scheme, c in [('random', '#3b6ea5'), ('speaker', '#c0392b')]:
    h = cnn_results[scheme]['hist']
    ax.plot(h['epoch'], h['val_macro_f1'], '-o', ms=3, color=c, label=f'{scheme} split')
ax.set(xlabel='epoch', ylabel='validation macro-F1', title='CNN learning curves')
ax.grid(alpha=.3)
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'sec9_cnn_curve.png'), dpi=130)
plt.show()
plt.close()

## ۹.۴ شکاف تعمیم و خطاهای صوتی

In [ ]:
summary = []
for scheme in ['random', 'speaker']:
    summary.append({
        'scheme': scheme,
        'svm_val': svm_results[scheme]['val_macro_f1'],
        'svm_test': svm_results[scheme]['test_macro_f1'],
        'cnn_val': cnn_results[scheme]['val_macro_f1'],
        'cnn_test': cnn_results[scheme]['test_macro_f1'],
    })
print(json.dumps(summary, indent=2))

### آزمایشِ بهبود: حذف ضریب انرژی (c0) از MFCC

`c0` لگاریتم انرژی کل فریم است و بیش از هر ضریب دیگری به **بلندی صدا و فاصله از
میکروفن** (یعنی هویت گوینده) وابسته است، نه به رقم. حذف آن باید تعمیم بین‌گوینده را
بهتر کند. این آزمایش فقط روی آموزش/اعتبارسنجی تصمیم گرفته می‌شود.

In [ ]:
F_nc0 = {name: summary_features(x, drop_c0=True) for name, x in waves.items()}
imp = {}
for scheme in ['random', 'speaker']:
    split = random_split if scheme == 'random' else speaker_split

    def mat(part, Fm):
        rows = split[part]
        return np.vstack([Fm[n] for n in rows['name']]), rows['digit'].to_numpy()

    (Xtr2, ytr2), (Xva2, yva2), (Xte2, yte2) = (mat('train', F_nc0),
                                                mat('val', F_nc0),
                                                mat('test', F_nc0))
    pipe2 = Pipeline([('scale', StandardScaler()),
                      ('clf', SVC(kernel='rbf', C=10, gamma='scale', random_state=SEED))])
    pipe2.fit(Xtr2, ytr2)
    imp[scheme] = {
        'val_macro_f1': float(f1_score(yva2, pipe2.predict(Xva2), average='macro')),
        'test_macro_f1': float(f1_score(yte2, pipe2.predict(Xte2), average='macro')),
    }
    print(f'SVM no-c0 [{scheme:7s}] VAL={imp[scheme]["val_macro_f1"]:.4f} '
          f'TEST={imp[scheme]["test_macro_f1"]:.4f}  '
          f'(baseline TEST={svm_results[scheme]["test_macro_f1"]:.4f})')

### حساسیت به گوینده: leave-one-speaker-out

با ۶ گوینده، هر بار یکی را کامل کنار می‌گذاریم (آموزش با ۵ گوینده) تا ببینیم
«سختی گوینده» چقدر متفاوت است و آیا افزودن گوینده کمک می‌کند.

In [ ]:
loso = {}
for s in sorted(df.speaker.unique()):
    tr_s, te_s = df[df.speaker != s], df[df.speaker == s]
    Xtr3 = np.vstack([F[n] for n in tr_s['name']])
    ytr3 = tr_s['digit'].to_numpy()
    Xte3 = np.vstack([F[n] for n in te_s['name']])
    yte3 = te_s['digit'].to_numpy()
    p3 = Pipeline([('scale', StandardScaler()),
                   ('clf', SVC(kernel='rbf', C=10, gamma=0.01, random_state=SEED))]).fit(Xtr3, ytr3)
    loso[s] = float(f1_score(yte3, p3.predict(Xte3), average='macro'))
    print(f'  held out {s:9s} -> macro-F1={loso[s]:.4f}')
print('LOSO mean %.4f | std %.4f | min %.4f | max %.4f'
      % (np.mean(list(loso.values())), np.std(list(loso.values())),
         min(loso.values()), max(loso.values())))

# سه خطای صوتی از آزمون گویندهٔ دیده‌نشده با مدل بهترِ آن طرح
best_scheme = max(['random', 'speaker'],
                  key=lambda s: max(svm_results[s]['test_macro_f1'],
                                    cnn_results[s]['test_macro_f1']))
use_cnn = cnn_results['speaker']['test_macro_f1'] >= svm_results['speaker']['test_macro_f1']
part = speaker_split['test']
if use_cnn:
    y_true, y_pred, _ = eval_cnn(cnn_results['speaker']['model'], part)
    who = 'CNN'
else:
    (Xt, yt), (Xv, yv), (Xe, ye) = build_X(speaker_split)
    y_true, y_pred = ye, svm_results['speaker']['test_pred']
    who = 'SVM'
print(f'\nerror examples from {who} on unseen-speaker test')

recall = {}
for d in range(10):
    m = y_true == d
    recall[d] = float((y_pred[m] == d).mean()) if m.any() else float('nan')
print('per-digit recall (unseen speaker):',
      {d: round(v, 2) for d, v in recall.items()})

# سه خطا با ترکیب (واقعی، پیش‌بینی) متفاوت تا تنوع خطاها دیده شود
wrong, seen_pairs = [], set()
for wi in np.where(y_true != y_pred)[0]:
    pair = (int(y_true[wi]), int(y_pred[wi]))
    if pair in seen_pairs:
        continue
    seen_pairs.add(pair)
    wrong.append(wi)
    if len(wrong) == 3:
        break

fig, axes = plt.subplots(2, 3, figsize=(11, 4.6))
for k, wi in enumerate(wrong):
    name = part['name'].iloc[wi]
    x = waves[name]
    axes[0, k].plot(np.arange(len(x)) / SR, x, lw=.8, color='#c0392b')
    axes[0, k].set_title(f'{name}\ntrue={y_true[wi]} pred={y_pred[wi]}', fontsize=8)
    axes[0, k].set_xlabel('time (s)', fontsize=8)
    L = logmel(x)
    axes[1, k].imshow(L, aspect='auto', origin='lower', cmap='magma')
    axes[1, k].set_xlabel('frame', fontsize=8)
    print(f'  {name}: true={y_true[wi]} pred={y_pred[wi]} dur={len(x)/SR:.2f}s')
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'sec9_errors.png'), dpi=130)
plt.show()
plt.close()

In [ ]:
res = {
    'n_files': len(df),
    'sample_rate': int(df.sr.iloc[0]),
    'duration': {'mean': round(float(durs.mean()), 3),
                 'min': round(float(durs.min()), 3),
                 'max': round(float(durs.max()), 3),
                 'median': round(float(np.median(durs)), 3)},
    'per_digit': dict(sorted(Counter(m['digit'] for m in meta).items())),
    'per_speaker': dict(Counter(m['speaker'] for m in meta)),
    'splits': {
        'random': {p: len(random_split[p]) for p in ['train', 'val', 'test']},
        'speaker': {p: len(speaker_split[p]) for p in ['train', 'val', 'test']},
    },
    'features': {'n_features': int(F[next(iter(F))].shape[0]),
                 'mfcc': N_MFCC, 'n_mels': N_MELS},
    'svm': {s: {k: v for k, v in svm_results[s].items()
                if k not in ('val_pred', 'val_true', 'test_pred', 'test_true')}
            for s in svm_results},
    'cnn': {s: {k: v for k, v in cnn_results[s].items()
                if k not in ('hist', 'model', 'test_true', 'test_pred')}
            for s in cnn_results},
    'summary': summary,
    'improvement_no_c0': imp,
    'loso': {k: round(v, 4) for k, v in loso.items()},
    'loso_mean': round(float(np.mean(list(loso.values()))), 4),
    'per_digit_recall_unseen': {int(d): round(v, 3) for d, v in recall.items()},
    'best_model_unseen': who,
}
with open(os.path.join(OUT, 'sec9_results.json'), 'w', encoding='utf-8') as f:
    json.dump(res, f, indent=2, ensure_ascii=False)
print('\nsaved out/sec9_results.json')

---

## بخش ۱۰ — ساخت عامل تقویتیِ کوچک با Q-learning

محیط خودساخته: شبکهٔ ۴×۴، شروع `(0,0)`، هدف `(3,3)`، چاله‌ها `(1,1)` و `(2,3)`.
اعمال: بالا/پایین/چپ/راست. برخورد با مرز، عامل را در همان خانه نگه می‌دارد.
پاداش: هر گام عادی −۰٫۰۱، هدف +۱، چاله −۱. هدف و چاله پایان اپیزودند و سقف هر
اپیزود ۴۰ گام است.

In [ ]:
import json
import os
import warnings

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import FancyArrowPatch, Rectangle

warnings.filterwarnings('ignore')
SEED = 42
OUT = 'out'
os.makedirs(OUT, exist_ok=True)

## ۱۰.۱ پیاده‌سازی محیط: `reset` و `step`

In [ ]:
class GridWorld:
    """شبکهٔ ۴×۴؛ حالت = (row, col) و عمل = ۰..۳."""

    N = 4
    START = (0, 0)
    GOAL = (3, 3)
    HOLES = {(1, 1), (2, 3)}
    ACTIONS = {0: (-1, 0), 1: (1, 0), 2: (0, -1), 3: (0, 1)}   # up, down, left, right
    ACTION_NAMES = {0: 'up', 1: 'down', 2: 'left', 3: 'right'}
    STEP_REWARD = -0.01
    GOAL_REWARD = 1.0
    HOLE_REWARD = -1.0
    MAX_STEPS = 40

    def __init__(self, max_steps=MAX_STEPS):
        self.max_steps = max_steps
        self.reset()

    def reset(self):
        self.pos = self.START
        self.t = 0
        self.done = False
        return self.pos

    def step(self, action):
        """یک گام؛ خروجی: (state بعدی، پاداش، پایان، اطلاعات)"""
        assert action in self.ACTIONS, f'invalid action {action}'
        self.t += 1
        dr, dc = self.ACTIONS[action]
        nr, nc = self.pos[0] + dr, self.pos[1] + dc

        # برخورد با مرز: ماندن در همان خانه
        hit_wall = not (0 <= nr < self.N and 0 <= nc < self.N)
        if hit_wall:
            nr, nc = self.pos
        self.pos = (nr, nc)

        if self.pos == self.GOAL:
            return self.pos, self.GOAL_REWARD, True, {'outcome': 'goal', 'steps': self.t,
                                                      'hit_wall': hit_wall}
        if self.pos in self.HOLES:
            return self.pos, self.HOLE_REWARD, True, {'outcome': 'hole', 'steps': self.t,
                                                      'hit_wall': hit_wall}
        if self.t >= self.max_steps:
            return self.pos, self.STEP_REWARD, True, {'outcome': 'timeout', 'steps': self.t,
                                                      'hit_wall': hit_wall}
        return self.pos, self.STEP_REWARD, False, {'outcome': 'step', 'steps': self.t,
                                                   'hit_wall': hit_wall}

    # ---- توابع کمکی برای کدگذاری حالت ----
    @classmethod
    def state_index(cls, pos):
        return pos[0] * cls.N + pos[1]

    @classmethod
    def index_state(cls, idx):
        return (idx // cls.N, idx % cls.N)

    @classmethod
    def is_terminal(cls, pos):
        return pos == cls.GOAL or pos in cls.HOLES


env = GridWorld()
N_STATES = env.N * env.N
N_ACTIONS = len(env.ACTIONS)
print('states:', N_STATES, '| actions:', N_ACTIONS)
print('start:', env.START, '| goal:', env.GOAL, '| holes:', env.HOLES)

### آزمون‌های کوچک: مرز، چاله، هدف و سقف گام

In [ ]:
tests = []

# ۱) حرکت از مرز: از (0,0) به بالا و چپ → ماندن در (0,0)
for a in [0, 2]:
    env.reset()
    s, r, d, info = env.step(a)
    tests.append({'test': f'move {GridWorld.ACTION_NAMES[a]} from (0,0)',
                  'state': s, 'reward': r, 'done': d, 'outcome': info['outcome']})

# ۲) رسیدن به هدف: مسیر (0,0)->(1,0)->(2,0)->(2,1)->(2,2)->(3,2)->(3,3)
path = [1, 1, 3, 3, 1, 3]
env.reset()
for a in path:
    s, r, d, info = env.step(a)
tests.append({'test': 'reach goal via ' + str(path), 'state': s,
              'reward': r, 'done': d, 'outcome': info['outcome'], 'steps': info['steps']})

# ۳) افتادن در چاله: (0,0)->(1,0)->(1,1)
env.reset()
env.step(1)
s, r, d, info = env.step(3)
tests.append({'test': 'fall into hole (1,1)', 'state': s,
              'reward': r, 'done': d, 'outcome': info['outcome'], 'steps': info['steps']})

# ۴) سقف گام: چرخیدن بین دو خانه برای ۴۰ گام
env = GridWorld(max_steps=5)
env.reset()
outs = [env.step(3 if i % 2 == 0 else 2) for i in range(5)]
s, r, d, info = outs[-1]
tests.append({'test': 'timeout after 5 dummy steps', 'state': s,
              'reward': r, 'done': d, 'outcome': info['outcome'], 'steps': info['steps']})

for t in tests:
    print(t)

## ۱۰.۲ Q-learning جدولی

$$Q(s,a) \leftarrow Q(s,a) + \alpha\big[r + \gamma \max_{a'} Q(s',a') - Q(s,a)\big]$$

برای حالت پایانی، مقدار هدف فقط `r` است (بدون بوت‌استرپ). سیاست ε-greedy با کاهش
خطی ε از ۱٫۰ تا ۰٫۰۵ در ۸۰٪ نخست آموزش و سپس ثابت روی ۰٫۰۵.

In [ ]:
ALPHA = 0.1
GAMMA = 0.95
N_EPISODES = 2000
EPS_START, EPS_END = 1.0, 0.05
EPS_DECAY_FRACTION = 0.8


def epsilon_at(ep):
    decay_eps = EPS_DECAY_FRACTION * N_EPISODES
    if ep >= decay_eps:
        return EPS_END
    return EPS_START + (EPS_END - EPS_START) * (ep / decay_eps)


def train_q_learning(n_episodes=N_EPISODES, alpha=ALPHA, gamma=GAMMA, seed=SEED):
    rng = np.random.RandomState(seed)
    Q = np.zeros((N_STATES, N_ACTIONS))
    hist = {'episode': [], 'reward': [], 'success': [], 'steps': [], 'epsilon': []}
    for ep in range(n_episodes):
        env = GridWorld()
        s = GridWorld.state_index(env.reset())
        eps = epsilon_at(ep)
        total, ok = 0.0, False
        while True:
            if rng.rand() < eps:
                a = rng.randint(N_ACTIONS)
            else:
                a = int(np.argmax(Q[s]))
            pos, r, done, info = env.step(a)
            s2 = GridWorld.state_index(pos)
            total += r
            if done:
                Q[s, a] += alpha * (r - Q[s, a])
                ok = (info['outcome'] == 'goal')
                break
            Q[s, a] += alpha * (r + gamma * Q[s2].max() - Q[s, a])
            s = s2
        hist['episode'].append(ep + 1)
        hist['reward'].append(total)
        hist['success'].append(int(ok))
        hist['steps'].append(info['steps'])
        hist['epsilon'].append(eps)
    return Q, hist


Q, hist = train_q_learning()
print(f'trained {N_EPISODES} episodes | alpha={ALPHA} gamma={GAMMA} '
      f'eps {EPS_START}->{EPS_END} | final eps={epsilon_at(N_EPISODES - 1):.3f}')
print('last 100 episodes: mean reward %.3f | success %.0f%%'
      % (np.mean(hist['reward'][-100:]), 100 * np.mean(hist['success'][-100:])))

## ۱۰.۳ ارزیابی: سیاست آموخته‌شده در برابر عامل تصادفی

۲۰۰ اپیزود **بدون اکتشاف** (فقط argmax) برای سیاست، و ۲۰۰ اپیزود با عمل یکنواخت
تصادفی برای عامل تصادفی. هر دو با بذر ثابت.

In [ ]:
def run_greedy(Q, n_episodes=200):
    rewards, successes, steps = [], [], []
    for _ in range(n_episodes):
        env = GridWorld()
        s = GridWorld.state_index(env.reset())
        total, ok = 0.0, False
        while True:
            a = int(np.argmax(Q[s]))
            pos, r, done, info = env.step(a)
            total += r
            if done:
                ok = (info['outcome'] == 'goal')
                break
            s = GridWorld.state_index(pos)
        rewards.append(total)
        successes.append(int(ok))
        steps.append(info['steps'])
    return np.array(rewards), np.array(successes), np.array(steps)


def run_random(n_episodes=200, seed=7):
    rng = np.random.RandomState(seed)
    rewards, successes, steps = [], [], []
    for _ in range(n_episodes):
        env = GridWorld()
        env.reset()
        total, ok = 0.0, False
        while True:
            _, r, done, info = env.step(rng.randint(N_ACTIONS))
            total += r
            if done:
                ok = (info['outcome'] == 'goal')
                break
        rewards.append(total)
        successes.append(int(ok))
        steps.append(info['steps'])
    return np.array(rewards), np.array(successes), np.array(steps)


g_rew, g_succ, g_steps = run_greedy(Q)
r_rew, r_succ, r_steps = run_random()
print(f'learned policy : mean reward={g_rew.mean():.4f} success={100 * g_succ.mean():.1f}% '
      f'mean steps={g_steps.mean():.1f}')
print(f'random policy  : mean reward={r_rew.mean():.4f} success={100 * r_succ.mean():.1f}% '
      f'mean steps={r_steps.mean():.1f}')
print(f'theoretical optimal return = 1 - 0.01*5 = 0.95 (path length 6)')

## سیاست نهایی و منحنی یادگیری

In [ ]:
fig, ax = plt.subplots(figsize=(5.6, 5.6))
arrow_dirs = {0: (0, 1), 1: (0, -1), 2: (-1, 0), 3: (1, 0)}  # up/down/left/right در مختصات نمودار
# سطر ۰ بالای نمودار است، پس «بالا» = +y
for r in range(4):
    for c in range(4):
        cell = (r, c)
        if cell == GridWorld.GOAL:
            color = '#2e7d32'
        elif cell in GridWorld.HOLES:
            color = '#8e2f2f'
        elif cell == GridWorld.START:
            color = '#3b6ea5'
        else:
            color = '#f2f2f2'
        ax.add_patch(Rectangle((c, 3 - r), 1, 1, facecolor=color, edgecolor='k', lw=1.2))
        if cell == GridWorld.GOAL or cell in GridWorld.HOLES or cell == GridWorld.START:
            label = 'GOAL' if cell == GridWorld.GOAL else ('HOLE' if cell in GridWorld.HOLES else 'START')
            ax.text(c + .5, 3 - r + .5, label, ha='center', va='center',
                    color='white', fontsize=9, weight='bold')
            continue
        q_row = Q[GridWorld.state_index(cell)]
        if np.allclose(q_row, 0.0):
            ax.text(c + .5, 3 - r + .5, '?', ha='center', va='center',
                    color='#888', fontsize=15, weight='bold')
            continue
        a = int(np.argmax(q_row))
        dx, dy = arrow_dirs[a]
        ax.add_patch(FancyArrowPatch((c + .5 - 0.28 * dx, 3 - r + .5 - 0.28 * dy),
                                     (c + .5 + 0.28 * dx, 3 - r + .5 + 0.28 * dy),
                                     arrowstyle='-|>', mutation_scale=16, lw=2,
                                     color='#1f3b57'))
ax.set_xlim(0, 4); ax.set_ylim(0, 4)
ax.set_xticks([]); ax.set_yticks([])
ax.set_title('Learned greedy policy (arrows = argmax Q)')
ax.set_aspect('equal')
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'sec10_policy.png'), dpi=130)
plt.show()
plt.close()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.4))
roll = 50
rew = np.array(hist['reward'], dtype=float)
succ = np.array(hist['success'], dtype=float)
eps_axis = np.array(hist['episode'])[roll - 1:]
ax.plot(eps_axis, np.convolve(rew, np.ones(roll) / roll, mode='valid'),
        color='#3b6ea5', label=f'reward (rolling {roll})')
ax2 = ax.twinx()
ax2.plot(eps_axis, np.convolve(succ, np.ones(roll) / roll, mode='valid'),
         color='#2e7d32', alpha=.8, label=f'success rate (rolling {roll})')
ax2.set_ylabel('success rate (rolling mean)', color='#2e7d32')
ax2.set_ylim(-0.02, 1.02)
ax.set(xlabel='episode', ylabel='episode return')
ax.set_title('Q-learning learning curve')
ax.grid(alpha=.3)
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc='lower right')
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'sec10_learning_curve.png'), dpi=130)
plt.show()
plt.close()

In [ ]:
fig, ax = plt.subplots(figsize=(5.4, 4.6))
vmax = float(np.abs(Q).max())
im = ax.imshow(Q.max(axis=1).reshape(4, 4), cmap='RdYlGn', vmin=-vmax, vmax=vmax)
for r in range(4):
    for c in range(4):
        ax.text(c, r, f'{Q.max(axis=1).reshape(4, 4)[r, c]:.2f}', ha='center', va='center', fontsize=8)
ax.set_xticks(range(4)); ax.set_yticks(range(4))
ax.set_title('max_a Q(s,a) per cell')
plt.colorbar(im, ax=ax, fraction=.046)
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'sec10_qvalues.png'), dpi=130)
plt.show()
plt.close()
print('figures ->', OUT)

In [ ]:
policy = {}
for r in range(4):
    for c in range(4):
        cell = (r, c)
        if cell in GridWorld.HOLES or cell == GridWorld.GOAL:
            continue
        a = int(np.argmax(Q[GridWorld.state_index(cell)]))
        policy[str(cell)] = GridWorld.ACTION_NAMES[a]
untrained = [GridWorld.index_state(i) for i in range(N_STATES)
             if not GridWorld.is_terminal(GridWorld.index_state(i))
             and np.allclose(Q[i], 0.0)]
print('states with Q == 0 (never visited):', untrained)
print('greedy actions:', json.dumps(policy, indent=0))

results = {
    'n_states': N_STATES,
    'n_actions': N_ACTIONS,
    'start': list(GridWorld.START),
    'goal': list(GridWorld.GOAL),
    'holes': [list(h) for h in GridWorld.HOLES],
    'rewards': {'step': GridWorld.STEP_REWARD, 'goal': GridWorld.GOAL_REWARD,
                'hole': GridWorld.HOLE_REWARD},
    'max_steps': GridWorld.MAX_STEPS,
    'tests': tests,
    'training': {'episodes': N_EPISODES, 'alpha': ALPHA, 'gamma': GAMMA,
                 'eps_start': EPS_START, 'eps_end': EPS_END,
                 'eps_decay_fraction': EPS_DECAY_FRACTION},
    'last100_train': {'mean_reward': round(float(np.mean(rew[-100:])), 4),
                      'success_rate': round(float(np.mean(succ[-100:])), 4)},
    'learned_policy': {'mean_reward': round(float(g_rew.mean()), 4),
                       'success_rate': round(float(g_succ.mean()), 4),
                       'mean_steps': round(float(g_steps.mean()), 2)},
    'random_policy': {'mean_reward': round(float(r_rew.mean()), 4),
                      'success_rate': round(float(r_succ.mean()), 4),
                      'mean_steps': round(float(r_steps.mean()), 2)},
    'policy_grid': policy,
    'untrained_states': [list(s) for s in untrained],
}
with open(os.path.join(OUT, 'sec10_results.json'), 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print('saved out/sec10_results.json')